<a href="https://colab.research.google.com/github/sonjoy1s/CNN_learn/blob/main/Cats_and_Dogs_image_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset , Subset
from torchvision import transforms
from PIL import Image
import kagglehub

In [13]:
#Download
path = kagglehub.dataset_download("samuelcortinhas/cats-and-dogs-image-classification")
print("path",path)

Using Colab cache for faster access to the 'cats-and-dogs-image-classification' dataset.
path /kaggle/input/cats-and-dogs-image-classification


In [14]:
torch.manual_seed(42)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [16]:
TRAIN_PATH = os.path.join(path,"/kaggle/input/cats-and-dogs-image-classification","train")
TEST_PATH = os.path.join(path,"/kaggle/input/cats-and-dogs-image-classification","test")

In [17]:
print(os.path.exists(TRAIN_PATH))
print(os.path.exists(TEST_PATH))

True
True


In [18]:
tranform = transforms.Compose(
    [
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5)*3,std=(0.5)*3)
    ]
)

In [19]:
from genericpath import isfile

class MyCustomeData(Dataset):
  def __init__(self, root_dir,transform=None) :
    super().__init__()

    self.samples = []
    self.transform = transform

    self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])

    self.class_to_idx= {cls_name : idx  for idx,cls_name in enumerate(self.classes)}

    for class_name in self.classes:
      class_path = os.path.join(root_dir,class_name)

      for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path,img_name)

        if os.path.isfile(img_path):
          label = self.class_to_idx[class_name]
          self.samples.append( (img_path,label) )


  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx) :

    img_path, label = self.samples[idx]
    image = Image.open(img_path).convert("RGB")

    if self.transform:
      image = self.transform(image)


    return image, label



In [20]:
train_dataset_full = MyCustomeData(TRAIN_PATH,tranform)
test_dataset_full = MyCustomeData(TEST_PATH,tranform)
num_classes = len(train_dataset_full.classes)

print("Number of classes:", num_classes)
print("Full train size:", len(train_dataset_full))
print("Full test size:", len(test_dataset_full))

Number of classes: 2
Full train size: 557
Full test size: 140


In [21]:
train_dataset = Subset(train_dataset_full, list(range (0, len(train_dataset_full))))
test_dataset = Subset(test_dataset_full, list(range(0, len(test_dataset_full))))

In [22]:
pin = True if device.type == 'cuda' else False
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=pin)

In [23]:
class MyCNNs(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*16*16, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [24]:
model = MyCNNs(num_classes=num_classes).to(device)

In [45]:
learning_rate = 0.0001
epochs = 10
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [46]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/10, Loss: 0.2696
Epoch 2/10, Loss: 0.2771
Epoch 3/10, Loss: 0.2675
Epoch 4/10, Loss: 0.2668
Epoch 5/10, Loss: 0.2695
Epoch 6/10, Loss: 0.2663
Epoch 7/10, Loss: 0.2625
Epoch 8/10, Loss: 0.2676
Epoch 9/10, Loss: 0.2658
Epoch 10/10, Loss: 0.2625


In [47]:
def evaluate(loader):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for batch_features, batch_labels in loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()
    return correct / total

In [48]:
test_acc = evaluate(test_loader)
print("Test Accuracy:", test_acc)
train_acc = evaluate(train_loader)
print("Train Accuracy:", train_acc)

Test Accuracy: 0.65
Train Accuracy: 1.0
